In [1]:
import pandas as pd
import re
import numpy as np
import iso639 

In [2]:
data = pd.read_csv('istc2.csv')
data['country'] = 'England'

In [3]:
print( data.shape )
print(f'The data set contains { data.shape[0]} rows. ')

(406, 13)
The data set contains 406 rows. 


In [4]:
print("Column headings:")

for column in data.columns.tolist():
    print(column)

Column headings:
author
dimensions
shelfmark
geonames
printer_place
printer
date
language
title
uncontrolled_term
identifier
probable_date
country


## Creating a subset

In [5]:
english_incunables = data.query('country=="England"')

## Analysis 

In [6]:
column = 'language'

all_values = data[column]
unique_values = data[column].dropna().unique()

for value in unique_values:
    #print(value)
    print(iso639.to_name(value))

English
French, Middle (ca.1400-1600)
Latin


In [7]:
iso639.to_name('lat')

'Latin'

* min()
* max()
* n()
* mean()

In [8]:
oldest = data['probable_date'].min()

In [34]:
## Group by

In [37]:
group_by_column = 'probable_date'
count_column = 'identifier'

def get_group_counts(df,group_by_column,count_column):
    ratio = df.groupby(group_by_column)[count_column].count()
    ratio = ratio.reset_index()
    ratio = ratio.rename(columns={count_column:'number'})
    return ratio

ratio = get_group_counts(data,group_by_column,count_column)

In [38]:
ratio

,probable_date,number
0,1468.0,1
1,1476.0,12
2,1477.0,9
3,1478.0,3
4,1479.0,4
5,1480.0,15
6,1481.0,15
7,1482.0,10
8,1483.0,24
9,1484.0,12


In [39]:
## A plot line

In [43]:
ratio = get_group_counts(data,'probable_date','identifier')

In [45]:

import plotly.express as px

fig = px.line(ratio , x="probable_date", y="number", title='Annual number of books')
fig.show()


# A Bar Chart

In [55]:
ratio = get_group_counts(data,'language','identifier')
ratio['language_name'] = ratio['language'].apply(iso639.to_name)

In [68]:
fig = px.bar(ratio, x="language_name", y="number", title='Annual number of books',color_discrete_sequence=['#7d0b05'])

fig.update_xaxes(tickangle=80)
fig.update_layout(
    xaxis_title="Date", yaxis_title="7 day avg")


fig.write_image("fig1.png")

# Interactive map

The data set that you have worked with has been georeferenced. The locations specified for the place of publication have been connected to geographic coordinates. These coordinates enable us to plot these locations on a map. 

This section of the tutorial concentrates on the visualisation of geospatial data using interactive maps using Leaflet and folium. Leaflet, firstly, is a widely used open source Javascript library for creating interactive maps. folium is a package built in the Python programming language, which makes the process of working with Leaflet much easier. You can view folium as an interface for Leaflet. 

If the folium package has not been installed yet, you can install it using the command below.


In [72]:
data

,author,dimensions,shelfmark,geonames,printer_place,printer,date,language,title,uncontrolled_term,identifier,probable_date,country
0,NaN,4°,PML 43328 (ChL 1816A),2634341.0,Westminster,Wynkyn de Worde,[about 1496],eng,Abbey of the Holy Ghost,NaN,"00RangeIndex(start=1, stop=407, step=1)",1496.0,England
1,NaN,4°,"IA.55232, Inc.5.J.1.2[3557]",2634341.0,Westminster,Wynkyn de Worde,[about 1497],eng,Abbey of the Holy Ghost,NaN,"00RangeIndex(start=1, stop=407, step=1)",1497.0,England
2,NaN,4°,Inc.335.,2634341.0,Westminster,Wynkyn de Worde,[about 1500],eng,Abbey of the Holy Ghost,NaN,"00RangeIndex(start=1, stop=407, step=1)",1500.0,England
3,NaN,f°,"15394.1, Ames I 49, C.12.i.10 = IB.55419, G.21...",2643743.0,[London,John Lettou and William de Machlinia,about 1481-82],frm,Abbreviamentum statutorum,NaN,"00RangeIndex(start=1, stop=407, step=1)",1481.0,England
4,NaN,8°,IA.55513,2643743.0,[London],Richard Pynson,9 Oct. 1499,frm,Abbreviamentum statutorum,NaN,"00RangeIndex(start=1, stop=407, step=1)",1499.0,England
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,NaN,f°,"IB.55422, Inc.3.J.3.2[3600]",2643743.0,[London,John Lettou and William de Machlinia,about 1482-83],frm,"Year book, 33 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1482.0,England
402,NaN,f°,"C.11.b.10 = IB.55469, G.6001 = IB.55469a, Inc....",2643743.0,London,William de Machlinia,[about 1485],frm,"Year book, 34 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1485.0,England
403,NaN,f°,"IB.55423, Inc.3.J.3.2[3601]",2643743.0,[London,John Lettou and William de Machlinia,about 1482],frm,"Year book, 35 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1482.0,England
404,NaN,f°,"IB.55424, Inc.3.J.3.2[3602]",2643743.0,[London,John Lettou and William de Machlinia,about 1482-83],frm,"Year Book, 36 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1482.0,England


In [83]:
coordinates = {2634341: ['51.4975', '-0.1357'],
 2643743: ['51.50853', '-0.12574'],
 2640729: ['51.75222', '-1.25596'],
 2638867: ['51.75', '-0.33333'],
 6269131: ['52.16045', '-0.70312']}


def add_latitude(g_id):
    if g_id is not np.nan and g_id is not pd.NA:
        return coordinates[g_id][0]
    else:
        return np.nan

def add_longitude(g_id):
    if g_id is not np.nan and g_id is not pd.NA:
        return coordinates[g_id][1]
    else:
        return np.nan
        

ratio = get_group_counts(data,'geonames','identifier')
ratio['latitude'] = ratio['geonames'].apply(add_latitude)
ratio['longitude'] = ratio['geonames'].apply(add_longitude)

In [77]:
import folium

In [102]:
centre = ratio.iloc[0]['latitude']
centre

'51.4975'

In [108]:


map_url = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}'
attribution = 'Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand), TomTom, 2012'

centre = ratio.iloc[0]


map = folium.Map(location=[ratio.iloc[0]['latitude'],ratio.iloc[0]['longitude']],
            tiles =  map_url , attr = attribution)

for i,row in ratio.iterrows():
    latitude = row['latitude']
    longitude = row['longitude']
    folium.Circle(
        location=[latitude,longitude],
        radius=3000,
        color='#1827c9',
        fill=False ,
        fill_opacity = 0.5,
        fill_color = '#c72d22' ,
        weight = 3 ,
        popup = f"{row['number']} books" ).add_to(map)
    
map.save('map.html')

In [109]:
map

In [69]:
In folium, You can make a very basic map, first of all, by creating a Map object.


folium.Map()


Once the map is created, you can write it to a file using the save() method. As a parameter for this method (i.e. the value supplied in beteween the brackets), you need to specify the filename. The code below will create a new file with the name ‘map.html’ in your current working directory.

map = folium.Map()
map.save('map.html')


To view the map, navigate to the folder containing the file named ‘map.html’, and doubleclick on this file. This file should then be opened in a web browser.

The default centre of this map is 0 latitude and 0 longitude. This is a location on the equator, with the latitude set to the location of the Greenwich observatory.

The centre of the map can be changed using the location parameter. The cell below places the centre at Picadilly Circus in London. The latitude of this location is 52.15775492822701, and the longitude in 4.482736320037433.


map = folium.Map()
map.save('map.html')


The zoom level can be adjusted using the `zoom_start` parameter. This zoom level defines the scale of the map. A low zoom level may show the full planet, or entire continents. A higher zoom level may show particular details of a location, such as the streets in a city. The defailt value for zoom_start is 0.



map = folium.Map( 
    location=[52.15723602487217, 4.482775926589967],
    zoom_start=12)
 



The base map that is used by default is provided by OpenStreetMap. You can also You can change the overall style of the map by using one of the following options in the tiles parameter of the Map.
CartoDB positron
CartoDB dark_matter
The width and the height of the map can also be adjusted. You can provide percentages for these properties.


map = folium.Map(location=[52.15723602487217, 4.482775926589967],
                 tiles='CartoDB positron',
 			width="50%",
height="100%",
zoom_start=12)
 




You can indicate a specific location on the map by added a Marker. As a parameter, you need to provide the coordinates of the location of this marker. More specifically, you need to specify the latitude and the longitude as a Python list, in square brackets.

You can also add a popup parameter to the Marker. When you do this, the text provided for the popup will become visible if it is clicked on.


map = folium.Map(location=[52.15723602487217, 4.482775926589967],
tiles = 'OpenStreetMap' ,
     width="90%",
    height="90%",
    zoom_start=15,   popup = 'London' )
 



Alternatively, you can work with the tooltipparameter. The text that is given will then appear when users hover over the marker.

A location can also be indicated using a Circle.
The appearance of such a circle can be manipulated using the following parameters.
radius changes the overall size of the circle.
weight determines the width of the border. The default width is 3.
color changes the colour of the border of the circle. To find colour codes, you can work with a color picker tool.
If you set fill to ‘True’, the interior of the circle will be coloured as well
fill_color determines the colour of the body of the circle. It can be different from the colour of the border.
You can work with the fill_opacityparameter to make the circle transparent. With value 0, the circle will be fully transparent. If the circle has an opacity of 1, there is no transparency.
As is the case for the marker, you can also add a popup.




map = folium.Map(location=[51.51366683783708, -0.09821931016133895],
                 tiles="OpenStreetMap",
width="%80",
height="%100",
zoom_start=14)

folium.Circle(
    location=[51.51366683783708, -0.09821931016133895],
    radius=120,
    color='#1827c9',
    fill=False ,
    fill_opacity = 0.5,
    fill_color = '#c72d22' ,
    weight = 3 ,
    popup = "St. Paul's Cathedral" ).add_to(map)




The size of a Circle is flexible. It is adjusted when user of the map zooms in or out. A CircleMarker, by contrast, will always have the radius that was specified originally.
The appearance of the Circle can be used with the exact same parameters.

As was discussed above, you can read CSV files using a library named pandas. The method named iterrows(), from Pandas, enables you iterate across all the rows in the dataframe. We can work with this specific function to add all the locations described in a CSV file as markers.




,author,dimensions,shelfmark,geonames,printer_place,printer,date,language,title,uncontrolled_term,identifier,probable_date,country
0,NaN,4°,PML 43328 (ChL 1816A),2634341.0,Westminster,Wynkyn de Worde,[about 1496],eng,Abbey of the Holy Ghost,NaN,"00RangeIndex(start=1, stop=407, step=1)",1496.0,England
1,NaN,4°,"IA.55232, Inc.5.J.1.2[3557]",2634341.0,Westminster,Wynkyn de Worde,[about 1497],eng,Abbey of the Holy Ghost,NaN,"00RangeIndex(start=1, stop=407, step=1)",1497.0,England
2,NaN,4°,Inc.335.,2634341.0,Westminster,Wynkyn de Worde,[about 1500],eng,Abbey of the Holy Ghost,NaN,"00RangeIndex(start=1, stop=407, step=1)",1500.0,England
3,NaN,f°,"15394.1, Ames I 49, C.12.i.10 = IB.55419, G.21...",2643743.0,[London,John Lettou and William de Machlinia,about 1481-82],frm,Abbreviamentum statutorum,NaN,"00RangeIndex(start=1, stop=407, step=1)",1481.0,England
4,NaN,8°,IA.55513,2643743.0,[London],Richard Pynson,9 Oct. 1499,frm,Abbreviamentum statutorum,NaN,"00RangeIndex(start=1, stop=407, step=1)",1499.0,England
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,NaN,f°,"IB.55422, Inc.3.J.3.2[3600]",2643743.0,[London,John Lettou and William de Machlinia,about 1482-83],frm,"Year book, 33 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1482.0,England
402,NaN,f°,"C.11.b.10 = IB.55469, G.6001 = IB.55469a, Inc....",2643743.0,London,William de Machlinia,[about 1485],frm,"Year book, 34 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1485.0,England
403,NaN,f°,"IB.55423, Inc.3.J.3.2[3601]",2643743.0,[London,John Lettou and William de Machlinia,about 1482],frm,"Year book, 35 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1482.0,England
404,NaN,f°,"IB.55424, Inc.3.J.3.2[3602]",2643743.0,[London,John Lettou and William de Machlinia,about 1482-83],frm,"Year Book, 36 Henry VI",NaN,"00RangeIndex(start=1, stop=407, step=1)",1482.0,England
